# Vector stores and semantic search



In [28]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Part I: Basic vector store implementation

In [29]:
class Document:
  def __init__(self, text: str, metadata: dict[str, str]):
    self.text = text
    self.metadata = metadata


class SearchResult:
  def __init__(self, score: float, document: Document):
    self.score = score
    self.document = document


class VectorStore:
  def __init__(self, embedding_model: SentenceTransformer):
    self.embedding_model = embedding_model
    self.documents = []
    self.embeddings = []

  def add_documents(self, documents: list[Document]):
    self.documents.extend(documents)
    texts = [doc.text for doc in documents]
    new_embeddings = self.embedding_model.encode(texts)
    self.embeddings.extend(new_embeddings)


  def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
    query_embedding = self.embedding_model.encode([query],convert_to_numpy=True)

    embeddings_matrix = np.array(self.embeddings)

    # Cos calculo
    similarities = cosine_similarity(
        query_embedding,
        embeddings_matrix
    )[0]

    # Ordenados indice y resultados
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for idx in top_indices:
      results.append(
          SearchResult(
              score=float(similarities[idx]),
              document=self.documents[idx]
          )
      )

    return results

In [30]:
df = pd.read_csv("animal-fun-facts-dataset.csv")
print(df)

           animal_name                                             source  \
0             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
1             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
2             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
3             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
4             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
...                ...                                                ...   
7729  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   
7730  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   
7731  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   
7732  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   
7733  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   

                                                   text media_link  \
0    

In [31]:
# Como una lista de instancias
documents = []

for _, row in df.iterrows():

    # De la clase Document proporcionada
    doc = Document(
        # Columna text es texto del documento
        text=str(row["text"]),

        # animal_name, source, media_link y wikipedia_link son metadatos
        metadata = {
        "animal_name": str(row["animal_name"]),
        "source": str(row["source"]),
        "media_link": str(row["media_link"]),
        "wikipedia_link": str(row["wikipedia_link"])
        }
    )

    documents.append(doc)

print(f"Total documentos: {len(documents)}")

Total documentos: 7734


In [ ]:
# Crear una instancia de VectorStore y agregar los documentos
model = SentenceTransformer('all-MiniLM-L6-v2')
vs_animals = VectorStore(model)
vs_animals.add_documents(documents)

In [33]:
# Realiza al menos 5 consultas de ejemplo y muestra los resultados obtenidos
queries = [
    "Animals that can fly",
    "Animals that live in water",
    "Mammals animals",
    "Animals that are carnivores",
    "Animals that use camouflage"
]

for q in queries:
  print("----------------------------------------------------------------------------")
  print(f"QUERY: {q}")
  results = vs_animals.search(q, top_k=3)

  # Incluye el score, texto y metadatos
  for r in results:
    print("Score:", r.score)
    print("Text:", r.document.text)
    print("Metadata:", r.document.metadata)
    print()

----------------------------------------------------------------------------
QUERY: Animals that can fly
Score: 0.7076399326324463
Text: They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.
Metadata: {'animal_name': 'colugo (flying lemur)', 'source': 'https://factanimal.com/colugo/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Colugo'}

Score: 0.6900621056556702
Text: Not all birds are able to fly!
Metadata: {'animal_name': 'bird', 'source': 'https://a-z-animals.com/animals/bird/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Bird'}

Score: 0.6480565667152405
Text: They rarely fly..
They move around on foot most of the time, only taking to the air to reach their nests or for courtship displays.
Metadata: {'animal_name': 'secretary bird', 'source': 'https://factanimal.com/secretarybird/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Secretarybird'}

--------------------------------------

## Part II: Filtering by metadata

In [90]:
class FilteredVectorStore:
  def __init__(self, embedding_model: SentenceTransformer):
    self.embedding_model = embedding_model
    self.documents = []
    self.embeddings = []

  def add_documents(self, documents: list[Document]):
    self.documents.extend(documents)
    texts = [doc.text for doc in documents]
    new_embeddings = self.embedding_model.encode(texts, convert_to_numpy=True)
    self.embeddings.extend(new_embeddings)

  def search(self, query: str,top_k: int = 5,metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
    query_embedding = self.embedding_model.encode([query],convert_to_numpy=True)

    filtered_docs = []
    filtered_embeddings = []

    for doc, emb in zip(self.documents, self.embeddings):

      include_doc = True

      # Si existe filtro por metadata
      if metadata_filter is not None:

        # Por cada filtro, si valor no coincide, no se incluye
        for key, value, in metadata_filter.items():
          if doc.metadata.get(key) != value:
            include_doc = False
            break

      # Filtrar los documentos según sus metadatos
      if include_doc:
        filtered_docs.append(doc)
        filtered_embeddings.append(emb)

    if len(filtered_docs) == 0:
      return []

    # Cos calculo
    similarities = cosine_similarity(query_embedding,np.array(filtered_embeddings))[0]

    # Ordenados indice y resultados
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for idx in top_indices:
      results.append(
          SearchResult(
              score=float(similarities[idx]),
              document=filtered_docs[idx]
          )
      )

    return results

In [48]:
# Dataset conformado por documentos de texto natural acompañados de metadatos
giftCards_df = pd.read_json(
    "gift_cards.json",
    lines=True
)

print(giftCards_df)

      overall  verified   reviewTime      reviewerID        asin  \
0           5      True  06 17, 2018  A31UBHTUUIFJUT  B004LLIKVU   
1           4      True  06 14, 2018  A2MN5JQMIY0FQ2  B004LLIKVU   
2           5      True   06 2, 2018  A25POI5IGGENPM  B004LLIKVU   
3           5      True  05 19, 2018  A2HYGTHB4LJ9FW  B004LLIKVU   
4           5      True  05 18, 2018   ACDG3M94UMZGJ  B004LLIKVU   
...       ...       ...          ...             ...         ...   
2967        5      True  07 10, 2018  A1MXZ1CW0ZVTKL  B01DWOZKSC   
2968        4      True  06 19, 2018  A1SVYJFIASQ46Z  B01DWOZKSC   
2969        5      True   06 8, 2018  A1QZ08NSDCZBA3  B01E4QS95I   
2970        5      True  11 10, 2017  A1L4GG3FBMIG6V  B01FERR9FW   
2971        5      True  07 10, 2018  A1MXZ1CW0ZVTKL  B01GKWEPBG   

                         style      reviewerName  \
0      {'Gift Amount:': ' 50'}      john stoiber   
1      {'Gift Amount:': ' 50'}   Amazon Customer   
2      {'Gift Amount:': ' 5

In [49]:
# Crea una instancia de FilteredVectorStore y agrega los documentos del dataset seleccionado.
gift_documents = []

for _, row in giftCards_df.iterrows():

    doc = Document(
      # combinamos summary + reviewText
      text=f"{row['summary']}. {row['reviewText']}",

      metadata = {
      "overall": str(row["overall"]),
      "verified": str(row["verified"]),
      "reviewTime": str(row["reviewTime"]),
      "reviewerID": str(row["reviewerID"]),
      "asin": str(row["asin"]),
      "reviewerName": str(row["reviewerName"])
      }
    )

    gift_documents.append(doc)

print(f"Total documentos: {len(gift_documents)}")

Total documentos: 2972


In [50]:
# Crear una instancia de FilteredVectorStore y agrega los documentos del dataset seleccionado
model = SentenceTransformer('all-MiniLM-L6-v2')
fvs_gift = FilteredVectorStore(model)
fvs_gift.add_documents(gift_documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [106]:
# Realiza al menos 5 consultas de ejemplo (con filtro de metadatos) y muestra los resultados obtenidos. Incluye el score, texto y metadatos.
# Consulta 1 con Filtro
q = "great gift products"

results = fvs_gift.search(
    query=q,
    top_k=3,
    metadata_filter={
         "verified": "True"
    }
)

print("QUERY:", q)

for r in results:
    print("Score:", r.score)
    print("Text:", r.document.text)
    print("Metadata:", r.document.metadata)
    print()

QUERY: great gift products
Score: 0.7506284713745117
Text: Great Gift Package. Nice gift packaging for any event
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '08 26, 2016', 'reviewerID': 'A35ZL3SVAXF593', 'asin': 'B014S24DAI', 'reviewerName': 'Jerry-Candymachines.com'}

Score: 0.7124781608581543
Text: great gift item. another favorite
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '11 7, 2016', 'reviewerID': 'A7VGY11A193XL', 'asin': 'B00BXLSZPM', 'reviewerName': 'Wendy'}

Score: 0.711276650428772
Text: Great gift idea. Product arrived on time and as described.
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '05 12, 2018', 'reviewerID': 'AYSOAWLV2LCCG', 'asin': 'B0091JKVU0', 'reviewerName': 'K'}



In [110]:
# Consulta 2 con Filtro
q = "bad gift products"

results_2 = fvs_gift.search(
    query=q,
    top_k=3,
    metadata_filter={
        "overall": "1"
    }
)

print("QUERY:", q)

for r in results_2:
    print("Score:", r.score)
    print("Text:", r.document.text)
    print("Metadata:", r.document.metadata)
    print()

QUERY: bad gift products
Score: 0.5091007351875305
Text: Do not buy ever. Nightmare. Avoid at all costs, Amazon and Wholefoods do not sell good gift cards online. Major hassle lots of embarrassing situations.
Metadata: {'overall': '1', 'verified': 'True', 'reviewTime': '08 21, 2016', 'reviewerID': 'A35TLFQDEJH95Y', 'asin': 'B00ELQD11E', 'reviewerName': 'R.K.Oceans'}

Score: 0.4924452602863312
Text: Garbage. Garbage delivery service. Avoid at all costs. Says 5min. All lies. Complications consistently with this gift card idea on amazon. Will nevet use regal again. Its been 30min already. Pathetic. Never an issue with carmike. This is why companys fail, poor service that i clearly got here.
Metadata: {'overall': '1', 'verified': 'False', 'reviewTime': '07 6, 2016', 'reviewerID': 'A35TLFQDEJH95Y', 'asin': 'B00MV9O08G', 'reviewerName': 'R.K.Oceans'}

Score: 0.4610651731491089
Text: REALLY!. you actually want a review on a gift card?!?!?!
Metadata: {'overall': '1', 'verified': 'True', 'revie

In [127]:
# Consulta 3 con Filtro
q = "excellent present for family"

results = fvs_gift.search(
    query=q,
    top_k=3,
    metadata_filter={
        "overall": "5"
    }
)

print("QUERY:", q)

for r in results:
    print("Score:", r.score)
    print("Text:", r.document.text)
    print("Metadata:", r.document.metadata)
    print()

QUERY: excellent present for family
Score: 0.6494894623756409
Text: GREAT XMAS PRESENT FOR ANYONE. Can't go wrong buying this for a present
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '07 7, 2016', 'reviewerID': 'A5U1KZ3EIK0IG', 'asin': 'B0091JKY0M', 'reviewerName': 'Branson Mom'}

Score: 0.6486921310424805
Text: Great Gift. Always a good gift for the family, the college kids especially.
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '01 7, 2017', 'reviewerID': 'ARLPBIM5RLZ64', 'asin': 'B00BXLSZPM', 'reviewerName': 'Marie in Ohio'}

Score: 0.6308112740516663
Text: This was a hit!. Gave to grand nieces and nephews as well as son and daughter.  Kept one for myself.  Everyone was very pleased.  Said they had not thought of giving this as a present.  Will do again.
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '01 18, 2014', 'reviewerID': 'A2QKY7ZG920FT7', 'asin': 'B00BXLSGHO', 'reviewerName': 'Jobie Austin'}



In [117]:
# Consulta 5 con Filtro
q = "situation with the gift card"

results = fvs_gift.search(
    query=q,
    top_k=3,
    metadata_filter={
        "overall": "3"
    }
)

print("QUERY:", q)

for r in results:
    print("Score:", r.score)
    print("Text:", r.document.text)
    print("Metadata:", r.document.metadata)
    print()

QUERY: situation with the gift card
Score: 0.5712001323699951
Text: Three Stars. it's a gift card
Metadata: {'overall': '3', 'verified': 'True', 'reviewTime': '01 16, 2017', 'reviewerID': 'A3INP3GS4178Y4', 'asin': 'B00GOLGWVK', 'reviewerName': 'Joseph A. Cordes'}

Score: 0.5491546392440796
Text: Three Stars. It is a gift card - the tag it self is ok
Metadata: {'overall': '3', 'verified': 'True', 'reviewTime': '01 14, 2018', 'reviewerID': 'ABWSFKJ1MRN1Q', 'asin': 'B01FERQPAM', 'reviewerName': 'Barb Peterson'}

Score: 0.5447947978973389
Text: OK, but Not Exactly as Expected. The gift card is obviously just fine, but the ribbon on the box is not the color pictured in the first picture; if you scroll down to the 5th picture, you'll see something more like the actual.  It is two or three shades lighter than shown so looks less classy, in my book.  The picture makes it look as if it is a deep, rich wine color.  And of course, the bow would have to be ironed to look as perky as in the picture

In [125]:
# Consulta 4 con Filtro
q = "excellent present for family"

results = fvs_gift.search(
    query=q,
    top_k=3,
    metadata_filter={
        "overall": "5"
    }
)

print("QUERY:", q)

for r in results:
    print("Score:", r.score)
    print("Text:", r.document.text)
    print("Metadata:", r.document.metadata)
    print()

QUERY: excellent present for family
Score: 0.6494894623756409
Text: GREAT XMAS PRESENT FOR ANYONE. Can't go wrong buying this for a present
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '07 7, 2016', 'reviewerID': 'A5U1KZ3EIK0IG', 'asin': 'B0091JKY0M', 'reviewerName': 'Branson Mom'}

Score: 0.6486921310424805
Text: Great Gift. Always a good gift for the family, the college kids especially.
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '01 7, 2017', 'reviewerID': 'ARLPBIM5RLZ64', 'asin': 'B00BXLSZPM', 'reviewerName': 'Marie in Ohio'}

Score: 0.6308112740516663
Text: This was a hit!. Gave to grand nieces and nephews as well as son and daughter.  Kept one for myself.  Everyone was very pleased.  Said they had not thought of giving this as a present.  Will do again.
Metadata: {'overall': '5', 'verified': 'True', 'reviewTime': '01 18, 2014', 'reviewerID': 'A2QKY7ZG920FT7', 'asin': 'B00BXLSGHO', 'reviewerName': 'Jobie Austin'}



##**Reflexión personal**

